## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [9]:
import os
from langchain.chat_models import init_chat_model
os.environ["Groq_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("openai/gpt-oss-20b",
    model_provider="groq",)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026AAF05F890>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026AAF1402D0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year: int = Field(description="This year the movie was realesed")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000026AAF05F890>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000026AAF1402D0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The titl

In [13]:
model.invoke("Provide details about the moview RRR")

AIMessage(content='**RRR (Rise Roar Revolt)** – a 2022 Indian Telugu-language epic action drama\n\n| Item | Details |\n|------|---------|\n| **Director** | S. S. Rajamouli (director of *Baahubali* series) |\n| **Production** | A. M. Rathnam, Y. G. Mahendra, S. S. Rajamouli (executive producers) |\n| **Production Companies** | Arka Media Works, 4 Lions Films |\n| **Release Date** | 13 March 2022 (India) – 15 March 2022 (USA & other international markets) |\n| **Runtime** | 161 minutes |\n| **Language** | Telugu (with dubbed versions in Tamil, Hindi, Malayalam, Kannada, Bengali, Marathi, Gujarati, and English) |\n| **Budget** | Approx. ₹300\u202fcr (₹3\u202fbillion) – one of the most expensive Indian films ever made |\n| **Box‑office** | Worldwide gross ≈ ₹2,200\u202fcr (₹220\u202fbillion) – record‑breaking for an Indian film in the first 30 days |\n| **Plot (high‑level)** | *RRR* is a fictional pre‑independence Indian story that follows two real‑life freedom fighters, **Komaram Bheem** 

In [16]:
response=model_with_structure.invoke("Provide details about the moview Kantara")
response

Movie(title='Kantara', year=2022, director='Rishab Shetty', rating=8.5)

In [17]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to provide details about the movie "Inception". The user didn\'t specify format. We can use the function to get movie details: we must call the function with director, rating, title, year. So we should call the function with appropriate arguments. The function returns a movie object with details. We\'ll need to supply rating out of 10, director, title, year. For Inception: Director: Christopher Nolan, Year: 2010, rating: maybe 8.8 (IMDB). We\'ll provide these. Use the function.', 'tool_calls': [{'id': 'fc_9ca40fa6-e15a-479d-a5c6-4e33ffba68ed', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 151, 'prompt_tokens': 164, 'total_tokens': 315, 'completion_time': 0.199784993, 'completion_tokens_details': {'reasoning_tokens': 112}, 'prompt_time': 0.008739977, 'pr

## Nested Structure

In [21]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        default=None,
        description="Budget in millions USD"
    )
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Inception."
)

print(response)

title='Inception' year=2010 cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')] genres=['Action', 'Adventure', 'Sci-Fi'] budget=160000000.0


In [22]:
print(response.title)
print(response.year)
print(response.cast)
print(response.genres)
print(response.budget)

Inception
2010
[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')]
['Action', 'Adventure', 'Sci-Fi']
160000000.0


In [20]:
response = model.invoke("Provide details about the movie Inception")

print(response.content)

**Inception (2010)**  
*A Christopher Nolan science‑fiction thriller that blends action, psychological drama, and mind‑bending concepts.*

---

## 1. Basic Production Information

| Item | Details |
|------|---------|
| **Director / Writer** | Christopher Nolan |
| **Producer** | Emma Thomas, Christopher Nolan, and others |
| **Screenplay** | Christopher Nolan |
| **Music** | Hans Zimmer (score) |
| **Cinematography** | Wally Pfister |
| **Editing** | Lee Smith |
| **Production Companies** | Syncopy, Warner Bros., Legendary Pictures |
| **Runtime** | 2 h 28 min (148 min) |
| **Country** | United States |
| **Language** | English |
| **Budget** | $160 million (approx.) |
| **Box‑Office** | $830 million worldwide |
| **Release Dates** | • 7 July 2010 – USA (limited)  <br>• 16 July 2010 – USA (wide)  <br>• 18 July 2010 – UK  <br>• 2010 – various international markets |
| **Distributor** | Warner Bros. Pictures |

---

## 2. Cast & Characters

| Actor | Character | Notes |
|-------|-------

## TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, 
ideal when you don’t need runtime validation.

In [23]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [27]:
from typing import TypedDict
from pydantic import Field
class Actor(TypedDict):
    name: str
    role: str
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Inception."
)
print("Title:", response["title"])
print("Year:", response["year"])
print("Cast:", response["cast"])
print("Genres:", response["genres"])
print("Budget:", response["budget"])

Title: Inception
Year: 2010
Cast: [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'}, {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'}, {'name': 'Elliot Page', 'role': 'Ariadne'}, {'name': 'Tom Hardy', 'role': 'Eames'}, {'name': 'Ken Watanabe', 'role': 'Saito'}, {'name': 'Cillian Murphy', 'role': 'Robert Fischer'}]
Genres: ['Action', 'Science Fiction', 'Thriller']
Budget: 160000000


## DataClasses


In [36]:
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_groq import ChatGroq
@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str
    email: str
    phone: str

model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": (
                "Extract contact info from: "
                "John Doe, john@example.com, (555) 123-4567"
            )
        }
    ]
})

print(result["structured_response"])

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')


In [37]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}